In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimplifiedContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(SimplifiedContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, hidden, y):
        # hidden = hidden.norm(dim=0,keepdim=True)
        cosine_similarity = F.cosine_similarity(hidden.unsqueeze(1), hidden.unsqueeze(0), dim=2)
        
        # 计算余弦距离
        pairwise_distances = 1.0 - cosine_similarity
        print(pairwise_distances)

        label_matrix = y.unsqueeze(1) == y.unsqueeze(0)
        
        positive_pairs = pairwise_distances * label_matrix.float()
        positive_loss = positive_pairs.pow(2).sum()

        negative_pairs = (self.margin - pairwise_distances) * (~label_matrix).float()
        negative_loss = F.relu(negative_pairs).pow(2).sum()
        print(negative_loss,positive_loss)
        loss = 0.5 * (positive_loss + negative_loss) / y.size(0)
        return loss
    

contrastive_criterion = SimplifiedContrastiveLoss()
inputs = torch.randn(8, 10)
labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])
loss = contrastive_criterion(inputs,labels)
print(loss)




In [ ]:
import torch.nn.functional as F

class ContrastiveCenterLoss(nn.Module):
    def __init__(self, dim_hidden, num_classes, lambda_c=1.0):
        super(ContrastiveCenterLoss, self).__init__()
        self.dim_hidden = dim_hidden
        self.num_classes = num_classes
        self.lambda_c = lambda_c
        self.centers = nn.Parameter(torch.randn(num_classes, dim_hidden))

    # may not work due to flowing gradient. change center calculation to exp moving avg may work.
    def forward(self, y, hidden):
        batch_size = hidden.size()[0]
        expanded_centers = self.centers.expand(batch_size, -1, -1)
        expanded_hidden = hidden.expand(self.num_classes, -1, -1).transpose(1, 0)
        distance_centers = (expanded_hidden - expanded_centers).pow(2).sum(dim=-1)
        distances_same = distance_centers.gather(1, y.unsqueeze(1))
        intra_distances = distances_same.sum()
        inter_distances = distance_centers.sum().sub(intra_distances)
        epsilon = 1e-6
        loss = (self.lambda_c / 2.0 / batch_size) * intra_distances / \
               (inter_distances + epsilon) / 0.1
        return loss
ContrastiveLoss = ContrastiveCenterLoss(10,4)
inputs = torch.randn(8, 10)
labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])
print(ContrastiveLoss(labels,inputs))

In [ ]:
get_query_mask(10)

In [ ]:
torch.triu(
            torch.full((5, 5),
                       float('-inf')), 1)

In [ ]:
!ls
from decoder import DecoderLayer, Decoder

In [ ]:
decoder_layer = DecoderLayer(384, 4, 384 * 4, 0.1)

In [ ]:
import torch.nn as nn
decoder = Decoder(decoder_layer, num_layers=1, norm=nn.LayerNorm(384))

In [ ]:
x_feat = torch.randn(20,1,384)

In [ ]:
attn_mask = torch.triu(
            torch.full((5,5),
                       float('-inf')), 1)
attn_mask

In [ ]:
def get_query_mask(n):
    matrix = torch.full((n, n), float('-inf'))
    torch.diagonal(matrix).fill_(0)
    matrix[:, 0] = 0
    return matrix

In [ ]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 

class ContrastiveLoss(nn.Module):
    def __init__(self, dim_hidden, num_classes, lambda_c=1.0):
        super(ContrastiveCenterLoss, self).__init__()
        self.dim_hidden = dim_hidden
        self.num_classes = num_classes
        self.centers = nn.Parameter(torch.randn(num_classes, dim_hidden))
    
    def forward(self,hidden,y):
        hidden = F.normalize(hidden,dim=-1)
        centers = F.normalize(self.centers,dim=-1)
        sim_martrix = hidden @ centers.T
        label_martrix = F.one_hot(y,num_classes = self.num_classes)
        loss = F.binary_cross_entropy_with_logits(sim_martrix,label_martrix)
        return loss
cen = ContrastiveLoss(dim_hidden=10,num_classes=4)
# ContrastiveLoss = ContrastiveCenterLoss(10,4)
inputs = torch.randn(8, 10)
labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])
print(cen(labels,inputs))

In [ ]:
def ContrastiveLoss(hidden, y, margin=1):
    hidden = F.normalize(hidden,dim=-1)
    sim_martrix =hidden @ hidden.T 
    # sim_martrix = F.cosine_similarity(hidden.unsqueeze(1), hidden.unsqueeze(0), dim=-1)
    label_matrix = y.unsqueeze(1) == y.unsqueeze(0)
    loss = F.binary_cross_entropy_with_logits(sim_martrix,label_matrix.float())
    
    return loss

In [ ]:
!pwd

In [ ]:
logits = []
tk_embedding = nn.Embedding(10, 384)
def decode(x_feat, tgt, query_mask, attn_mask,tgt_padding_mask=None):
    B, N, C = x_feat.shape
    null_ctx = tk_embedding(tgt[:, :1])
    tgt_emb = tk_embedding(tgt[:, 1:])
    tgt_emb = torch.cat([x_feat, null_ctx, tgt_emb], dim=1)
    print(tgt_emb.shape)
    return decoder(tgt_emb,query_mask,attn_mask,tgt_padding_mask)   


tgt_out = decode(x_feat,tgt_in[:,:3],query_mask[:4,:4],attn_mask[:4,:4])


In [ ]:
tgt_out.shape